# Export W&B Project

This notebook exports everything useful from the W&B project:

- run configs
- run summaries
- run metadata
- full unsampled metric histories via `scan_history()`
- files attached to each run
- optional artifacts

Project path used here:

```text
usi-cv/sentiment-project
```

In [7]:
# Run this once if needed
%pip install wandb pandas tqdm pyarrow

Note: you may need to restart the kernel to use updated packages.


In [8]:
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
import wandb

/opt/homebrew/Caskroom/miniforge/base/envs/dll/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Login

Run the cell below and paste your W&B API key if prompted.

In [9]:
wandb.login()

True

## Configuration

In [10]:
ENTITY = "usi-cv"
PROJECT = "sentiment-project"
OUT = Path("wandb_export")

OUT.mkdir(exist_ok=True)
api = wandb.Api()

## Fetch runs

In [11]:
runs = list(api.runs(f"{ENTITY}/{PROJECT}"))
print(f"Found {len(runs)} runs")

Found 14 runs


## Export all runs

This exports one folder per run under `wandb_export/runs/<run_id>/`.

In [12]:
summary_rows = []

for run in tqdm(runs):
    rdir = OUT / "runs" / run.id
    files_dir = rdir / "files"
    files_dir.mkdir(parents=True, exist_ok=True)

    # Config, summary, metadata
    (rdir / "config.json").write_text(
        json.dumps(dict(run.config), indent=2, default=str)
    )
    (rdir / "summary.json").write_text(
        json.dumps(dict(run.summary), indent=2, default=str)
    )
    (rdir / "metadata.json").write_text(
        json.dumps({
            "id": run.id,
            "name": run.name,
            "path": run.path,
            "state": run.state,
            "created_at": str(run.created_at),
            "url": run.url,
            "tags": run.tags,
            "notes": run.notes,
            "sweep": run.sweep.id if run.sweep else None,
        }, indent=2, default=str)
    )

    # Full unsampled metric history
    history = list(run.scan_history())
    if history:
        pd.DataFrame(history).to_csv(rdir / "history.csv", index=False)

    # Files attached to the run
    for f in run.files():
        try:
            f.download(root=str(files_dir), replace=True)
        except Exception as e:
            print("File failed:", run.id, f.name, e)

    summary_rows.append({
        "id": run.id,
        "name": run.name,
        "state": run.state,
        "url": run.url,
        **dict(run.summary),
    })

runs_summary = pd.DataFrame(summary_rows)
runs_summary.to_csv(OUT / "runs_summary.csv", index=False)
runs_summary

100%|██████████| 14/14 [03:13<00:00, 13.83s/it]


,id,name,state,url,Confusion Matrix,Grad-CAM Heatmaps,Learning Curves (Plot),Train Accuracy,Train F1,Train Loss,Val Accuracy,Val F1,Val Loss,Val Recall,_runtime,_step,_timestamp,_wandb,epoch,learning_rate
0,05lsoc3f,Run_mobilenet_fer2013,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.877565,0.874389,0.348844,0.664531,0.654245,1.189045,0.637793,2200.000000,27,1.779020e+09,{'runtime': 2200},26,0.00050
1,inv7b30q,Run_mobilenet_fer2013,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.877565,0.874389,0.348844,0.664531,0.654245,1.189045,0.637793,2212.000000,27,1.779023e+09,{'runtime': 2212},26,0.00050
2,mfpnvefy,Run_mobilenet_fer2013,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.877565,0.874389,0.348844,0.664531,0.654245,1.189045,0.637793,2210.000000,27,1.779025e+09,{'runtime': 2210},26,0.00050
3,0mdo7vzo,Run_inception_fer2013,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.845902,0.835715,0.437393,0.663555,0.636434,1.100854,0.634487,2614.000000,25,1.779027e+09,{'runtime': 2614},24,0.00050
4,q6bgavya,Run_inception_fer2013,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.845902,0.835715,0.437393,0.663555,0.636434,1.100854,0.634487,2606.000000,25,1.779030e+09,{'runtime': 2606},24,0.00050
5,d32r93s6,Run_inception_fer2013,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.845902,0.835715,0.437393,0.663555,0.636434,1.100854,0.634487,2631.000000,25,1.779033e+09,{'runtime': 2631},24,0.00050
6,40afx6wo,Run_mobilenet_ckplus,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.987245,0.981271,0.036626,0.934010,0.930855,0.158223,0.936226,124.000000,46,1.779033e+09,{'runtime': 124},45,0.00050
7,9szmvxl6,Run_mobilenet_ckplus,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.987245,0.981271,0.036626,0.934010,0.930855,0.158223,0.936226,120.000000,46,1.779033e+09,{'runtime': 120},45,0.00050
8,dhwalzsl,Run_mobilenet_ckplus,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",0.987245,0.981271,0.036626,0.934010,0.930855,0.158223,0.936226,120.000000,46,1.779033e+09,{'runtime': 120},45,0.00050
9,a9u2gawo,Run_inception_ckplus,finished,https://wandb.ai/usi-cv/sentiment-project/runs...,"{'_type': 'image-file', 'format': 'png', 'heig...","{'_type': 'images/separated', 'captions': ['Tr...","{'_type': 'image-file', 'format': 'png', 'heig...",1.000000,1.000000,0.002782,0.984772,0.981572,0.059885,0.975673,225.000000,64,1.779033e+09,{'runtime': 225},63,0.00005


## Optional: export artifacts

Run this only if the project actually uses W&B Artifacts.

In [13]:
artifacts_dir = OUT / "artifacts"
artifacts_dir.mkdir(exist_ok=True)

artifact_rows = []

for artifact_type in api.artifact_types(project=PROJECT, entity=ENTITY):
    for collection in artifact_type.collections():
        for artifact_version in collection.versions():
            target = artifacts_dir / artifact_type.name / collection.name / artifact_version.version
            target.mkdir(parents=True, exist_ok=True)
            try:
                artifact_version.download(root=str(target))
                artifact_rows.append({
                    "type": artifact_type.name,
                    "collection": collection.name,
                    "version": artifact_version.version,
                    "name": artifact_version.name,
                    "path": str(target),
                })
            except Exception as e:
                print("Artifact failed:", artifact_version.name, e)

artifacts_summary = pd.DataFrame(artifact_rows)
artifacts_summary.to_csv(OUT / "artifacts_summary.csv", index=False)
artifacts_summary

CommError: Api.artifact_types() got an unexpected keyword argument 'entity'

## Zip the export folder

In [ ]:
import shutil

zip_path = shutil.make_archive("wandb_export", "zip", OUT)
zip_path